In [ ]:
# No-reference image quality evaluation for UmbraLift.
#
# PIQE, NIQE and BRISQUE are all "lower is better" and need no ground truth,
# which is what makes them usable here: there are no clean reference images
# for a permanently shadowed region.

import os

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from pyiqa import create_metric

import model
from utils.device import get_device, describe

In [ ]:
def load_network(snapshot_path, device):
    """Build the network and load weights once, rather than per image."""
    net = model.umbra_lift_net().to(device)
    net.load_state_dict(torch.load(snapshot_path, map_location=device, weights_only=True))
    net.eval()
    return net


def to_uint8(tensor_image):
    """Tensor -> HWC uint8, matching what a saved PNG would contain."""
    arr = tensor_image.squeeze(0).permute(1, 2, 0).cpu().detach().numpy()
    return (np.clip(arr, 0.0, 1.0) * 255).astype(np.uint8)


def load_image(path, device):
    """Load an image, cropping to a multiple of 8 for the pooling stack."""
    img = Image.open(path).convert("RGB")
    width = (img.size[0] // 8) * 8
    height = (img.size[1] // 8) * 8
    if (width, height) != img.size:
        img = img.resize((width, height))
    arr = np.asarray(img) / 255.0
    tensor = torch.from_numpy(arr).float().permute(2, 0, 1)
    return tensor.to(device).unsqueeze(0)


def lowlight_compare(folder_path, snapshot="snapshots/model-best.pth", preview=None):
    """
    Enhance every image in a folder and report mean PIQE / NIQE / BRISQUE.

    Args:
        folder_path: directory of low-light images.
        snapshot: checkpoint to evaluate.
        preview: filename to display before/after; defaults to the first image.
    """
    device = get_device()
    print(f"Running on {describe(device)}")

    # Metrics and network are built once, outside the loop.
    metrics = {name: create_metric(name) for name in ("piqe", "niqe", "brisque")}
    net = load_network(snapshot, device)

    scores = {name: [] for name in metrics}
    preview_pair = None

    filenames = sorted(
        f for f in os.listdir(folder_path)
        if f.lower().endswith((".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"))
    )
    if not filenames:
        raise RuntimeError(f"No images found in {folder_path}")

    with torch.no_grad():
        for filename in filenames:
            data_lowlight = load_image(os.path.join(folder_path, filename), device)
            enhanced, _ = net(data_lowlight)

            # Round-trip through uint8 so the metrics see what would be saved.
            enhanced_np = to_uint8(enhanced)
            enhanced_tensor = torch.tensor(enhanced_np).permute(2, 0, 1).unsqueeze(0).float() / 255.0

            for name, metric in metrics.items():
                scores[name].append(metric(enhanced_tensor).item())

            if preview_pair is None and (preview is None or filename == preview):
                preview_pair = (to_uint8(data_lowlight), enhanced_np, filename)

    averages = {name: float(np.mean(values)) for name, values in scores.items()}

    print(f"\nAverage scores over {len(filenames)} image(s) (lower is better):")
    print(f"PIQE: {averages['piqe']:.2f}, NIQE: {averages['niqe']:.2f}, "
          f"BRISQUE: {averages['brisque']:.2f}")

    if preview_pair is not None:
        original, enhanced_np, filename = preview_pair
        fig, axes = plt.subplots(1, 2, figsize=(12, 6))
        axes[0].imshow(original)
        axes[0].set_title(f"Input — {filename}")
        axes[0].axis("off")
        axes[1].imshow(enhanced_np)
        axes[1].set_title("Enhanced")
        axes[1].axis("off")
        plt.tight_layout()
        plt.show()

    return averages

In [ ]:
averages = lowlight_compare("data/test_data/PSR/")